In [9]:
# Main
import numpy as np
from scipy import stats
import pandas as pd

# Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# Split and Cross Val
from sklearn.model_selection import train_test_split, GridSearchCV

# Preprocess and Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.svm import SVC
from sklearn.cluster import KMeans

# Ensemble Models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier

# Metrics
from sklearn.metrics import (mean_absolute_error, 
                             accuracy_score, 
                             root_mean_squared_error, 
                             r2_score, 
                             mean_absolute_error,
                             mean_squared_error, 
                             silhouette_score)

# Environments and serialization
import joblib

# Datasets
from sklearn.datasets import load_diabetes

In [12]:
heavy_df = pd.read_csv('data/spacex_web_scraped.csv')

In [13]:
heavy_df.head()

,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Date,Time,Version Booster,Booster landing
0,1,Cape Canaveral,Dragon Spacecraft Qualification Unit,U,LEO,SpaceX,Success,4 June 2010,18:45,F9v1.0,Failure (parachute)
1,2,Cape Canaveral,SpaceX COTS Demo Flight 1,U,LEO,NASA,Success,8 December 2010,15:43,F9v1.0,Failure (parachute)
2,3,Cape Canaveral,SpaceX COTS Demo Flight 2,525 kg,LEO,NASA,Success,22 May 2012,07:44,F9 v1.0,No attempt
3,4,Cape Canaveral,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,8 October 2012,00:35,F9 v1.0,No attempt
4,5,Cape Canaveral,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,1 March 2013,15:10,F9 v1.0,No attempt


In [14]:
heavy_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 521 entries, 0 to 520
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Flight No.       521 non-null    int64 
 1   Launch site      521 non-null    object
 2   Payload          518 non-null    object
 3   Payload mass     521 non-null    object
 4   Orbit            519 non-null    object
 5   Customer         507 non-null    object
 6   Launch outcome   521 non-null    object
 7   Date             521 non-null    object
 8   Time             521 non-null    object
 9   Version Booster  521 non-null    object
 10  Booster landing  521 non-null    object
dtypes: int64(1), object(10)
memory usage: 44.9+ KB


In [15]:
heavy_df['Launch outcome'].value_counts()

Launch outcome
Success    519
Failure      2
Name: count, dtype: int64

In [16]:
heavy_df['Booster landing'].value_counts()

Booster landing
Success ( OCISLY           161
Success ( JRTI             121
Success ( ASOG             117
Success ( LZ‑1              38
Success ( LZ‑4              26
No attempt                  18
No attempt                   9
Success ( LZ‑2               7
Controlled (ocean)           6
Failure ( OCISLY             5
Failure ( JRTI               4
Failure (parachute)          2
Failure (ocean)              2
Failure ( LZ‑1               1
Precluded  (drone ship)      1
Success( JRTI                1
Failure ( ASOG               1
Success ( LZ‑40              1
Name: count, dtype: int64

In [15]:
heavy_df["site_flag"] = None

# Cape = 1
heavy_df.loc[
    heavy_df["Launch site"].str.contains("Cape Canaveral|Kennedy", case=False, na=False),
    "site_flag"
] = 1

# Vandenberg = 0
heavy_df.loc[
    heavy_df["Launch site"].str.contains("Vandenberg", case=False, na=False),
    "site_flag"
] = 0

In [21]:
heavy_df.sample(50)

,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Date,Time,Version Booster,Booster landing,site_flag
313,405,Cape Canaveral,Starlink,"~17,500 kg",LEO,SpaceX,Success,"December 4, 2024",10:13,F9B5,Success ( ASOG,1
265,357,Vandenberg,Starlink,"~16,500 kg",LEO,SpaceX,Success,"July 28, 2024",09:22,F9B5,Success ( OCISLY,0
93,94,Kennedy,Starlink,"15,600 kg",LEO,SpaceX,Success,6 October 2020,11:29:34,F9 B5[,Success ( OCISLY,1
341,433,Kennedy,WorldView Legion,"1,500 kg",LEO,Maxar Technologies,Success,"February 4, 2025",23:13,F9B5,Success ( LZ‑1,1
415,507,Vandenberg,TRACERS,~920 kg,SSO,NASA,Success,"July 23, 2025",18:13,F9B5,Success ( LZ‑4,0
40,41,Kennedy,Boeing X-37B,"4,990 kg",LEO,USAF,Success,7 September 2017,14:00,F9B4[,Success ( LZ‑1,1
447,539,Cape Canaveral,Starlink,"~16,100 kg",LEO,SpaceX,Success,"September 25, 2025",08:39,F9B5,Success ( ASOG,1
327,419,Cape Canaveral,Starlink,"~17,500 kg",LEO,SpaceX,Success,"January 6, 2025",20:43,F9B5,Success ( JRTI,1
59,60,Cape Canaveral,Merah Putih,"5,800 kg",GTO,Telkom Indonesia,Success,7 August 2018,05:18,F9B5[,Success ( OCISLY,1
167,168,Cape Canaveral,Danuri (Korea Pathfinder Lunar Orbiter),~679 kg,BLT,KARI,Success,4 August 2022,23:08,F9 B5,Success ( JRTI,1
